# 鲁棒验证：让模型检查模型

> 上一讲我们让模型对同一道题多次采样，再用 self-consistency 对最终答案做多数投票。投票只统计答案出现的次数，不检查每条解的推理对不对——当错误答案扎堆时，多数投票会选出错的答案。
>
> 这一节引入独立的验证器：它给每条候选解打一个"正确概率"，测试时用 best-of-N 挑选最高分的解。我们从 Cobbe 的结果验证器讲起，经过 Lightman 的过程验证器、Math-Shepherd 的无人工标注，最后把验证器与采样结合，并讨论多个弱验证器的集成。


训练好的模型无法在生成过程中回头修改自己的推理：自回归逐 token 输出，一步算错，后续步骤全部建立在错误之上。多步数学题对单点错误尤其敏感。生成侧的这个弱点，恰好对应验证侧的机会——判断一条已经生成的解对不对，比从头推导它容易。

验证器（verifier）是一条独立的评分通路：给定一条候选解，输出一个 0 到 1 之间的正确概率。它不负责生成，只负责评判，而且可以独立于生成器训练。这正是本节的核心：生成容易，验证难，而验证恰恰是把采样出来的正确解挑出来的关键。

差距可以用两个量表达。`Pass@K` 指候选池里存在至少一条正确解的比例，即"正确答案有没有被生成出来"；`SuccessRate` 指选择器实际挑对的概率。两者的差就是生成-验证差距（generation-verification gap）。我们先用一个可控的玩具任务测出这两个量。


## 1. 生成与验证的差距

自回归生成没有纠错机制。模型逐步输出 token，某个中间步骤一旦算错，后续推理就都建立在错误之上，错误答案扎堆时多数投票也会失效。Weaver 论文把这个问题量化成两个指标：Pass@K 与 SuccessRate，两者的差就是生成-验证差距。

我们把"答案唯一、可程序判对错"这一性质编码成一个玩具任务：每道题由两个子表达式和一个组合算子构成，最终答案是两个子结果组合得到的值。解文本的格式固定，便于按行解析和重算。


## 2. 验证器的训练：Cobbe 的数学验证器

OpenAI 的 Cobbe 等人在发布 GSM8K 的同时提出了验证器的训练流程。流程分三段：先在训练题上微调生成器两个 epoch；再对每道题从生成器采样 100 条候选解，按"最终答案是否等于标准答案"自动打正负标签；最后在这个自动标签集上训练一个独立的验证器（一个 epoch）。

验证器与生成器是两个独立的网络。验证器在每条解的末尾输出一个正确概率（solution-level）；也可以在每个 token 之后都预测一次整条解的正确概率（token-level），测试时取最后一个 token 的分数。后者相当于一条 token 级价值函数，也预示了后文的过程监督。

自动打标有一个明显的噪声源：中间步算错、最终答案却碰巧正确的解（歪打正着），会被标成正例；中间步全对、最终答案写错的解，会被标成负例。我们用玩具任务把这份噪声统计出来。


## 3. 结果验证器与过程验证器

Cobbe 的结果验证器只用最终答案对不对做监督：每条解一个标签。Lightman 等人指出这种做法有信用分配问题——一条负标签只说明"某处错了"，无法指出错在哪一步；难题的候选解大多含错，负标签的边际信息量很低。

过程验证器（PRM，process reward model）把监督粒度推进到每一步：对推理链的每一步给出一个正确概率，这样"哪一步拉低了整条解"的信息被直接编码进标签。下表是四种组合：

| 中间步 | 最终答案 | 结果监督标签 | 过程监督标签 |
|---|---|---|---|
| 全对 | 对 | 正例 | 每步都正 |
| 有错 | 对（歪打正着） | 正例 | 含一个负步 |
| 全对 | 错（收尾写错） | 负例 | 每步都正 |
| 有错 | 错 | 负例 | 含负步 |

注意中间两行：结果监督把"中间错、答案对"标成正例，把"中间对、答案错"标成负例，标签与推理质量脱节；过程监督则直接给出每一步的真值。


## 4. Step-by-step 验证：Lightman

Lightman 等人（2023）把验证从判结果推进到判过程。过程验证器对推理链的每一步输出该步正确的概率，整条解的分数用乘积归约（所有步都正确的概率），neutral 标签按正例处理。

在 MATH 数据集的 best-of-1860 上，PRM 达到 78.2%，结果验证器 ORM 为 72.4%，多数投票 69.6%，且差距随候选数 N 增大而拉大。PRM800K 是 80 万条人工步级标注，只标到第一个错误步为止，兼顾成本与信息量。

我们在玩具任务上复现这条主线：训练两个验证器，一个用结果标签（每解一条），一个用过程标签（每步一条），比较它们学出来的分数在 best-of-N 里的表现。这里我们把第 2 节用的一致性特征还原为原始数值，让两个验证器面对同样的信息量。


## 5. 无人工标注的步级验证：Math-Shepherd

过程监督需要人工标注每一步，PRM800K 有 80 万条标签，成本很高。Math-Shepherd（Wang 等人，2023）用自动标注替代人工：给定推理链的某个中间步，用一个补全器（completer）从该步续写 N 条后续路径，按这些路径最终是否到达正确答案来给这一步打标。思路是 MCTS 式的：某一步的质量，由"从这一步继续还能不能推出正确答案"决定。

两种估计方式。硬估计（HE）只要有一条续写到达正确答案，该步就标为 1；软估计（SE）用到达正确答案的续写比例。记第 j 条续写的最终答案为 $a_j$，黄金答案为 $a^*$：

```text
HE: y_i = 1[存在 j 使 a_j = a*]
SE: y_i = (1/N) * Σ_j 1[a_j = a*]
```

作者发现 SE 的分布更接近人工标注，HE 在续写数 N 增大时因为假阳性而退化。我们用 mock 补全器复现这个规律。


## 6. 验证器与采样结合

测试时，验证器与采样结合构成 best-of-N：采样 N 条候选，验证器打分，选最高分。Cobbe 还发现对打分最高的 top-k 条解再做一次多数投票通常更好；Lightman 与 Math-Shepherd 都报告验证器与 self-consistency 组合优于任一方。

单个弱验证器不够可靠时，可以把多个验证器加权集成。Weaver（Saad-Falcon 等人，2025）用弱监督把多个弱验证器的精度估计出来，再按朴素贝叶斯聚合：假设各验证器在解正确性条件下独立，每个验证器有真阳性率（TPR）与真阴性率（TNR），它们两两之间的同意率可以反推出这些精度——不需要人工标签，用矩估计就能解出来。


验证信号可以按来源排成一条序列：程序真值检查（执行器/最终答案）→ 结果监督（整条标签）→ 过程监督（每步标签）→ 多验证器集成（无标签）。成本递增，鲁棒性也递增。下一讲我们看另一种更硬的验证信号——把推理放进工具里执行，以执行结果为准；第 6 讲的强化学习会把本讲的验证器当作 reward 信号，reward 的质量直接决定训练的上限。


## 小结

- [ ] 生成容易、验证难：自回归生成没有纠错机制，中间一步出错会污染整条解
- [ ] 生成-验证差距 = Pass@K − SuccessRate，多数投票在错误答案扎堆时失效
- [ ] Cobbe 的验证器训练流程：生成器采样 → 按最终答案自动打标 → 训练独立验证器 → 测试时 best-of-N
- [ ] 自动结果标签有两类噪声：歪打正着（中间错、答案对）与收尾写错（中间对、答案错）
- [ ] 结果监督（ORM）用整条标签，存在信用分配问题；过程监督（PRM）给每步一个标签
- [ ] PRM 的整条分数用乘积或最小值归约，在 best-of-N 里明显优于多数投票与结果验证器
- [ ] Math-Shepherd 用补全器自动标注步级标签，HE 在 N 大时引入假阳性，SE 更稳
- [ ] 多个弱验证器可用矩估计做无标签加权集成，逼近 oracle 的选解能力

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

三道填空题都附带参考答案，单元格可直接执行；建议先遮住答案，自己写一遍再对照。

**作业 1：PRM 解分数的两种归约**

给定一条解每步的正确概率数组，补全乘积归约与最小值归约。

小提示：乘积归约把所有步的概率乘起来，步数越多分数越小；最小值归约只看最差的一步，哪一步错了，它会立刻暴露。


**作业 2：Math-Shepherd 的 HE 与 SE**

给定一条中间步的 N 条续写路径最终是否正确，补全硬估计与软估计。

小提示：HE 是存在性（任一为 True 即 1），SE 是频率（True 的比例）。N 越小 SE 噪声越大。

**作业 3：验证器选解 vs 多数投票**

给定一组候选，每条带（验证器分数，最终答案，是否正确）。补全"按分数选最高分"与"按答案多数投票"两种选择器。

小提示：多数投票只统计答案出现次数，不关心每条解的验证分数；当错误答案扎堆但分数很低时，验证器胜出。


## 参考资料

- Cobbe et al., [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168), 2021 — 验证器与 GSM8K 的奠基论文，确立结果验证与 best-of-N 框架
- Lightman et al., [Let's Verify Step by Step](https://arxiv.org/abs/2305.20050), 2023 — 结果监督与过程监督的系统比较，发布 PRM800K
- Wang et al., [Math-Shepherd: Verify and Reinforce LLMs Step-by-step without Human Annotations](https://arxiv.org/abs/2312.08935), 2023 — 用补全器自动标注步级标签（HE/SE）并做 step-by-step PPO
- Saad-Falcon et al., [Shrinking the Generation-Verification Gap with Weak Verifiers](https://arxiv.org/abs/2506.18203), NeurIPS 2025 — 无标签矩估计与弱验证器集成（Weaver）
- Uesato et al., [Solving Math Word Problems with Process- and Outcome-based Feedback](https://arxiv.org/abs/2211.14275), 2022 — outcome 与 process 的第一篇对比
- Li et al., [Making Language Models Better Reasoners with Step-Aware Verifier](https://arxiv.org/abs/2210.01241), 2022 — NLI/规则自动步标注方法，Math-Shepherd 的基线
- Wang et al., [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171), 2022 — 多数投票基线（上一讲已介绍）
- Hendrycks et al., [Measuring Mathematical Problem Solving with the MATH Dataset](https://arxiv.org/abs/2103.03874), 2021 — MATH 数据集
- Shao et al., [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/abs/2402.03300), 2024 — GRPO，step-level reward 在强化学习中的延续

这个玩具任务里一道题长什么样，先用具体例子看清楚。题目由两个子表达式和一个组合算子构成。
题面：3 + 4 与 2 * 5，组合算子为 +。黄金答案按两步算出：先算 3 + 4 = 7、2 * 5 = 10，再算 7 + 10 = 17。
一条候选解是固定格式的三行文本，前两行是两个子表达式的结果，第三行是最终答案：

```text
3 + 4 = 7
2 * 5 = 10
答案 = 17
```

这条解好不好，可以逐行重算判断：第一行重算 3 + 4 是否等于 7，第二行重算 2 * 5 是否等于 10，第三行看最终答案是否等于黄金答案 17。于是每条解都能得到两个布尔数组：每步是否算对（step_ok）与最终答案是否写对（final_ok）。
答案唯一、可程序判对错，是整个玩具任务的关键性质，它让我们能自动给任意一条解打标签，不必请人看。下面用代码随机生成这样的题和解，每道题的黄金答案由程序唯一确定。


In [ ]:
import numpy as np
import re

OPS = {"+": lambda a, b: a + b, "-": lambda a, b: a - b, "*": lambda a, b: a * b}


def apply(op, a, b):
    """执行一步算术运算，返回整数结果。"""
    return OPS[op](a, b)


def make_problem(rng):
    """随机生成一道两段式算术题，返回题面、组合算子与黄金答案。"""
    a1 = int(rng.integers(1, 10))
    op1 = str(rng.choice(["+", "-", "*"]))
    b1 = int(rng.integers(1, 10))
    if op1 == "-" and a1 < b1:
        a1, b1 = b1, a1
    a2 = int(rng.integers(1, 10))
    op2 = str(rng.choice(["+", "-", "*"]))
    b2 = int(rng.integers(1, 10))
    if op2 == "-" and a2 < b2:
        a2, b2 = b2, a2
    cop = str(rng.choice(["+", "-"]))
    golden = apply(cop, apply(op1, a1, b1), apply(op2, a2, b2))
    return {"parts": [(a1, op1, b1), (a2, op2, b2)],
            "combine": cop, "golden": golden}


def slip(rng):
    """生成一个小幅随机算术偏差（±1 到 ±3）。"""
    return int(rng.integers(1, 4)) * int(rng.choice([-1, 1]))


def sample_solution(problem, step_acc, rng):
    """用可调准确率的 mock 答题器生成一条候选解。

    每个中间步以 step_acc 概率算对；最终答案的正确性由中间步推导决定，
    再叠加少量端部噪声。返回 (文本, 每步对错, 最终对错, 最终答案值)。
    """
    (a1, op1, b1), (a2, op2, b2) = problem["parts"]
    cop = problem["combine"]
    golden = problem["golden"]
    l_true = apply(op1, a1, b1)
    r_true = apply(op2, a2, b2)
    l_p = l_true if rng.random() < step_acc else l_true + slip(rng)
    r_p = r_true if rng.random() < step_acc else r_true + slip(rng)
    derived = (l_p == l_true) and (r_p == r_true)
    eps, lucky = 0.08, 0.08
    final_ok = (derived and rng.random() > eps) or \
               (not derived and rng.random() < lucky)
    if final_ok:
        a_p = golden
    else:
        a_p = golden - 1 if rng.random() < 0.8 else golden + slip(rng)
    text = "\n".join([
        f"{a1} {op1} {b1} = {l_p}",
        f"{a2} {op2} {b2} = {r_p}",
        f"答案 = {a_p}",
    ])
    step_ok = [l_p == l_true, r_p == r_true]
    return text, step_ok, final_ok, a_p


rng = np.random.default_rng(42)
for _ in range(2):
    p = make_problem(rng)
    text, step_ok, final_ok, _ = sample_solution(p, 0.5, rng)
    print("黄金答案:", p["golden"])
    print(text)
    print("每步对错:", step_ok, "| 最终对错:", final_ok)
    print("---")

In [ ]:
def judge_solution(text, problem):
    """把解文本按行切成步骤，逐行重算真值并与黄金答案比对。

    返回 (step_ok, final_ok)：每步是否正确、最终答案是否正确。
    """
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    (a1, op1, b1), (a2, op2, b2) = problem["parts"]
    golden = problem["golden"]
    expect = [apply(op1, a1, b1), apply(op2, a2, b2), golden]
    step_ok = []
    for line in lines:
        m = re.match(r"^(-?\d+)\s*([+\-*])\s*(-?\d+)\s*=\s*(-?\d+)$", line)
        if m is None:
            continue
        i = len(step_ok)
        c = int(m.group(4))
        step_ok.append(c == expect[i])
    final_ok = None
    for line in lines:
        m = re.match(r"^答案 = (-?\d+)$", line)
        if m:
            final_ok = int(m.group(1)) == golden
    return step_ok, final_ok


rng = np.random.default_rng(7)
for _ in range(2):
    p = make_problem(rng)
    text, _, _, _ = sample_solution(p, 0.5, rng)
    step_ok, final_ok = judge_solution(text, p)
    print("黄金答案:", p["golden"])
    print(text)
    print("程序判定: 每步", step_ok, "| 最终", final_ok)
    print("---")

Pass@K 与 SuccessRate 的差别，用一道题、三条解就能说清楚。
假设 K = 3，对同一道题（黄金答案 17）采出三条解：

```text
解 A：3 + 4 = 7，2 * 5 = 10，答案 = 17   （最终正确）
解 B：3 + 4 = 8，2 * 5 = 10，答案 = 18   （最终错误）
解 C：3 + 4 = 8，2 * 5 = 10，答案 = 18   （最终错误）
```

候选池里有正确解：解 A 在池子里，所以 Pass@3 = 1，正确的答案确实被生成出来了。
把三条解按最终答案做多数投票：答案 18 出现两次、17 出现一次，投票选出 18，而正确答案是 17。这次选择失败了，SuccessRate（就这一题）为 0。
Pass@3 − SuccessRate = 1 − 0 = 1，这就是生成-验证差距。这个差距不是生成能力的损失，正确答案已经在池子里，而是选择环节的损失：选择器只看答案出现次数，被扎堆的错误答案盖过。验证器要补的就是这个位置——不数次数，而是评估每条解本身的质量。


In [ ]:
def extract_answer(text):
    """从解文本里取出最终答案数字。"""
    for line in text.strip().splitlines():
        m = re.match(r"^答案 = (-?\d+)$", line.strip())
        if m:
            return int(m.group(1))
    return None


def pass_at_k(demos, K, step_acc, seed=1):
    """对每题采样 K 条解，统计池子里至少有一条最终答案正确的比例。"""
    rng = np.random.default_rng(seed)
    ok = 0
    for p in demos:
        hit = any(sample_solution(p, step_acc, rng)[2] for _ in range(K))
        ok += int(hit)
    return ok / len(demos)


def majority_success(demos, K, step_acc, seed=1):
    """对每题采样 K 条解，多数投票选出现最多的答案，统计命中率。"""
    from collections import Counter
    rng = np.random.default_rng(seed)
    ok = 0
    for p in demos:
        votes = [sample_solution(p, step_acc, rng)[3] for _ in range(K)]
        top = Counter(votes).most_common(1)[0][0]
        ok += int(top == p["golden"])
    return ok / len(demos)


rng = np.random.default_rng(42)
demos = [make_problem(rng) for _ in range(40)]
K = 25
pk = pass_at_k(demos, K, step_acc=0.5)
mj = majority_success(demos, K, step_acc=0.5)
print("Pass@K（池子里有正确解）: {:.2f}".format(pk))
print("多数投票 SuccessRate:      {:.2f}".format(mj))
print("生成-验证差距:             {:.2f}".format(pk - mj))

Cobbe 的验证器训练流程可以拆成三步，每一步都只依赖自动判断，中途不需要人。
整个流程始于生成器：先用训练数据微调生成器，让它能解出训练题。之后进入三步。
第一步，采样。对每个训练题，用训练好的生成器采 100 条候选解。生成器有随机性，同一道题每次采出的解可能不同。
第二步，自动打标。每条解只有一个标签：最终答案等于黄金答案就标 1，否则标 0。这一步只比较一个整数，完全自动化。以前面例子里的三条解为例，解 A 标 1，解 B、解 C 标 0。
第三步，训练验证器。把（解的特征，标签）当成普通监督学习数据，训练一个独立的网络去预测这条解最终正确的概率。测试时这个验证器对新的候选解打分。
采样、打标、训练全部自动，这是结果监督（outcome supervision）的定义：监督信号只有整条解的最终结果。自动打标不是没有噪声，中间步算错、最终答案碰巧等于黄金答案的解（歪打正着）会被误标成正例；中间步全对、最后一行写错的解会被误标成负例。下面把这两种噪声在数据里统计出来。


In [ ]:
rng = np.random.default_rng(42)
n_probs = 200
cpp = 25
step_acc = 0.5
problems = [make_problem(rng) for _ in range(n_probs)]

pool = []
for p in problems:
    for _ in range(cpp):
        text, step_ok, final_ok, a = sample_solution(p, step_acc, rng)
        pool.append(dict(prob=p, text=text, step_ok=step_ok,
                         final_ok=final_ok, answer=a))

n_clean = sum(1 for d in pool if all(d["step_ok"]) and d["final_ok"])
n_lucky = sum(1 for d in pool if not all(d["step_ok"]) and d["final_ok"])
n_misend = sum(1 for d in pool if all(d["step_ok"]) and not d["final_ok"])
print("候选总数:", len(pool))
print("中间步全对且最终答案对:", n_clean, "({:.2f})".format(n_clean / len(pool)))
print("中间步有错但最终答案对（歪打正着）:", n_lucky,
      "({:.2f})".format(n_lucky / len(pool)))
print("中间步全对但最终答案错（收尾写错）:", n_misend,
      "({:.2f})".format(n_misend / len(pool)))

# 展示一条"歪打正着"的解
lucky_demo = next(d for d in pool if not all(d["step_ok"]) and d["final_ok"])
print("\n歪打正着示例:")
print(lucky_demo["text"])
print("黄金答案:", lucky_demo["prob"]["golden"])

In [ ]:
def step_consistency(text):
    """解析文本里每一步的字面一致性（重算 A op B == C）。"""
    cons = []
    for line in text.strip().splitlines():
        line = line.strip()
        m = re.match(r"^(-?\d+)\s*([+\-*])\s*(-?\d+)\s*=\s*(-?\d+)$", line)
        if m:
            a, op, b, c = (int(m.group(1)), m.group(2),
                           int(m.group(3)), int(m.group(4)))
            cons.append(1.0 if apply(op, a, b) == c else 0.0)
    return cons


def orm_features(text):
    """结果验证器的输入特征：每步一致性 + 答案值 + 常量。

    每步一致性标记由解析重算得到，相当于验证头顺手检查了每步算术。
    """
    cons = step_consistency(text)
    ans = 0.0
    for line in text.strip().splitlines():
        m = re.match(r"^答案 = (-?\d+)$", line.strip())
        if m:
            ans = int(m.group(1)) / 100.0
    return np.array(cons[:2] + [ans, 1.0], dtype=np.float32)


n_tr = 140
tr_mask = np.zeros(len(pool), bool)
for i in range(n_tr):
    tr_mask[i * cpp:(i + 1) * cpp] = True
te_mask = ~tr_mask

X = np.stack([orm_features(d["text"]) for d in pool])
Yf = np.array([1.0 if d["final_ok"] else 0.0 for d in pool], dtype=np.float32)
Xtr, Ytr, Xte, Yte = X[tr_mask], Yf[tr_mask], X[te_mask], Yf[te_mask]
print("训练集:", Xtr.shape, "测试集:", Xte.shape)


import torch
import torch.nn as nn

torch.manual_seed(0)


def make_mlp(d_in, hidden=32):
    """构造三层全连接网络，输出一个 logit。"""
    return nn.Sequential(
        nn.Linear(d_in, hidden), nn.ReLU(),
        nn.Linear(hidden, hidden // 2), nn.ReLU(),
        nn.Linear(hidden // 2, 1),
    )


def train_mlp(model, Xt_, Yt_, iters=2000, batch=256, lr=0.005):
    """在 (特征, 标签) 上训练 MLP，返回逐迭代的损失列表。"""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    Xt = torch.tensor(Xt_)
    Yt = torch.tensor(Yt_).unsqueeze(1)
    n = len(Xt)
    losses = []
    for _ in range(iters):
        idx = torch.randperm(n)[:batch]
        opt.zero_grad()
        loss = loss_fn(model(Xt[idx]), Yt[idx])
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


def predict_prob(model, X_):
    """用模型对特征矩阵打分，返回 sigmoid 后的概率。"""
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(X_))).numpy().ravel()


orm = make_mlp(4, hidden=32)
losses = train_mlp(orm, Xtr, Ytr, iters=2000)
p_orm = predict_prob(orm, Xte)
acc = ((p_orm > 0.5) == Yte).mean()
print("结果验证器测试准确率: {:.3f}".format(acc))
print("损失首尾: {:.3f} -> {:.3f}".format(losses[0], losses[-1]))

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("BCE loss")
plt.title("Result verifier training loss")
plt.tight_layout()
plt.show()

验证器训练好之后，测试时与采样结合成 best-of-N：对同一道题采 N 条候选解，验证器给每条打一个正确概率，选分数最高的那条的答案。
挑最高分而不是多数投票，原因在于两者的判断对象不同。多数投票只统计答案出现次数，不检查每条解的推理质量；当错误答案扎堆时（很多条解在同一处犯同一种错），投票会把错误的答案选出来。验证器给每条解单独打分，分数低的解即使答案出现次数多也选不中。回顾前面的三条解：多数投票选 18，一个理想的验证器会给解 A 高分、给解 B 和 C 低分，最终选解 A。
Cobbe 还观察到一种增强：先按验证器分数取 top-k 条解，再对这 k 条解做最终答案多数投票，通常比单纯选最高分更好。最高分那一条可能只是轻微占优，而 top-k 里正确答案往往反复出现。Lightman 与 Math-Shepherd 都报告验证器与 self-consistency 组合优于任何一方单独使用。
下面用同一个训练好的结果验证器做 best-of-N，与多数投票对比。候选解来自测试题，验证器没有在测试题上训练过。


In [ ]:
te_start = n_tr * cpp
te_answers = np.array([d["answer"] for d in pool])
te_golden = np.array([d["prob"]["golden"] for d in pool])
te_final_ok = np.array([1.0 if d["final_ok"] else 0.0 for d in pool])
n_te = (len(pool) - te_start) // cpp
print("测试题数:", n_te)


def best_of_n_success(N, score_fn):
    """对每个测试题取前 N 条候选，按分数挑最高分，统计最终答案命中率。

    score_fn(base, N) 返回第 base 个测试题前 N 条候选的分数数组。
    """
    ok = 0
    for pi in range(n_te):
        base = pi * cpp
        idx = base + int(np.argmax(score_fn(base, N)))
        ok += int(te_answers[te_start + idx] == te_golden[te_start + idx])
    return ok / n_te


def majority_success(N):
    """多数投票：取前 N 条候选，选出现最多的答案。"""
    ok = 0
    for pi in range(n_te):
        base = te_start + pi * cpp
        votes = te_answers[base:base + N]
        counts = {}
        for v in votes:
            counts[v] = counts.get(v, 0) + 1
        top = max(counts, key=counts.get)
        ok += int(top == te_golden[base])
    return ok / n_te


for N in [10, 25]:
    mj = majority_success(N)
    orm_ok = best_of_n_success(N, lambda b, N: p_orm[b:b + N])
    print("N={}: 多数投票 {:.2f} | 结果验证器 best-of-N {:.2f}".format(
        N, mj, orm_ok))

结果验证器（ORM）与过程验证器（PRM）的区别，用同一道题、同一条解演示一遍。
题目仍是 3 + 4 与 2 * 5 组合相加，黄金答案 17。模型给出一条解：

```text
3 + 4 = 8    （第一步算错，应为 7）
2 * 5 = 10   （第二步算对）
答案 = 18
```

结果监督只看最终答案：最终答案不等于黄金答案，整条解打一个标签 0。结果验证器对这条解的判断就落在这一个分数位上——一个 0 或 1，只说明最终结果错了，没说错在哪一步。
过程监督把标签推进到每一步：第一步算错标 0，第二步算对标 1，这条解得到两个步级标签 [0, 1]，标签直接指出了错误的位置。训练时，结果验证器每个样本一条解一个标签，过程验证器每个样本一个步骤一个标签，这就是两者监督粒度的差别。
评分时，两种验证器都要把分数归约成整条解的一个数。过程验证器给每步输出一个正确概率，例如 [0.9, 0.8, 0.2, 0.9]（第三步是错的）。乘积归约把所有步的概率乘起来：0.9 × 0.8 × 0.2 × 0.9 = 0.1296，含义是所有步都正确的概率，任一步概率低都会压垮整条分数。最小值归约取所有步概率的最小值：min(0.9, 0.8, 0.2, 0.9) = 0.2，含义是最差的一步决定整条解的分数，哪一步最可疑，分数立刻暴露它。
两种归约都依赖同一件事：每步分数携带了错在哪的信息。这正是结果监督缺失的——整条解一个标签，无法区分第一步错、第二步对和第一步对、第二步错，这个信息缺失就是信用分配问题。下面先用代码算两种归约，再在一条真实的歪打正着解上比较两种监督的标签。


In [ ]:
def score_product(step_probs):
    """乘积归约：把所有步的正确概率相乘，作为整条解的分数。"""
    return float(np.prod(step_probs))


def score_min(step_probs):
    """最小值归约：取所有步正确概率的最小值。"""
    return float(np.min(step_probs))


step_probs = np.array([0.9, 0.8, 0.2, 0.9])
print("四条步概率:", step_probs)
print("乘积归约:", round(score_product(step_probs), 4))
print("最小值归约:", round(score_min(step_probs), 4))

all_ok = np.array([0.9, 0.8, 0.9, 0.9])
print("\n全对时 乘积归约:", round(score_product(all_ok), 4),
      "| 最小值归约:", round(score_min(all_ok), 4))

In [ ]:
# 挑一条"中间步有错、最终答案对"的解，比较两种监督的标签
lucky = next(d for d in pool[te_start:] if not all(d["step_ok"])
             and d["final_ok"])
print("题面:", lucky["prob"]["parts"], "| 组合:",
      lucky["prob"]["combine"], "| 黄金答案:", lucky["prob"]["golden"])
print(lucky["text"])
step_ok, final_ok = judge_solution(lucky["text"], lucky["prob"])
print("每步真值:", step_ok, "| 最终真值:", final_ok)

true_step_probs = np.array(step_ok, dtype=float)
print("结果监督标签: +1（最终答案对）")
print("过程监督标签:", [1.0 if s else 0.0 for s in step_ok])
print("完美过程验证器 乘积归约:", round(score_product(true_step_probs), 3))
print("完美结果验证器（已知最终对错）: 1.0")

结果监督只有整条标签，下面的实验检验它能否从标签中学出错误位置。
前面训练的结果验证器用了每步一致性标记作为特征——手工把每步重算是否相等算好喂给它，相当于提前把过程信息塞进了特征。现在去掉这些标记，只给它每步的操作数、算子、结果这些原始数值，加上整条答案，再训练同一个结构的验证器。
实验会显示，只有整条标签时，验证器很难学出错误位置。原因在标签本身的信息量：一条负标签只说明这条解某处错了，没说错在第一步还是第二步。同结构、同训练轮数，学不到错误位置，准确率就掉到接近全部猜错的基线。
这就是信用分配问题——整条标签把错误位置的信息稀释了。过程监督把标签粒度切到每步，把错在哪直接写进标签，绕开了这个问题。


In [ ]:
OP_CODE = {"+": 0, "-": 1, "*": 2}


def step_features(text, k):
    """第 k 步的数值特征：操作数与结果（归一化）+ 算子独热 + 步号。"""
    line = [ln.strip() for ln in text.strip().splitlines() if ln.strip()][k]
    m = re.match(r"^(-?\d+)\s*([+\-*])\s*(-?\d+)\s*=\s*(-?\d+)$", line)
    a, op, b, c = int(m.group(1)), m.group(2), int(m.group(3)), int(m.group(4))
    onehot = [0.0, 0.0, 0.0]
    onehot[OP_CODE[op]] = 1.0
    return [a / 50.0, b / 50.0, c / 50.0] + onehot + [k / 3.0]


def raw_solution_features(text):
    """结果验证器的原始数值特征：两步特征 + 答案 + 常量，没有一致性标记。"""
    feats = []
    for k in range(2):
        feats += step_features(text, k)[:6]
    ans = 0.0
    for line in text.strip().splitlines():
        m = re.match(r"^答案 = (-?\d+)$", line.strip())
        if m:
            ans = int(m.group(1)) / 100.0
    return np.array(feats + [ans, 1.0], dtype=np.float32)


Xr = np.stack([raw_solution_features(d["text"]) for d in pool])
Xr_tr, Xr_te = Xr[tr_mask], Xr[te_mask]
print("原始特征维度:", Xr_tr.shape[1])

orm_raw = make_mlp(14, hidden=64)
_ = train_mlp(orm_raw, Xr_tr, Ytr, iters=2500)
p_raw = predict_prob(orm_raw, Xr_te)
acc_raw = ((p_raw > 0.5) == Yte).mean()
print("结果验证器（原始特征）测试准确率: {:.3f}".format(acc_raw))
print("基线（全预测错）: {:.3f}".format(1 - Yte.mean()))
print("关键观察: 只有整条标签时，验证器学不到'哪一步错了'。")

In [ ]:
def raw_step_dataset():
    """把每条解的每个步展开成独立的 (特征, 标签) 样本。"""
    Xs_, ys_ = [], []
    for d in pool:
        for k in range(2):
            Xs_.append(step_features(d["text"], k))
            ys_.append(1.0 if d["step_ok"][k] else 0.0)
    return np.array(Xs_, np.float32), np.array(ys_, np.float32)


Xs_all, ys_all = raw_step_dataset()
mask_rep = np.repeat(tr_mask, 2)
Xs_tr, ys_tr = Xs_all[mask_rep], ys_all[mask_rep]
Xs_te, ys_te = Xs_all[~mask_rep], ys_all[~mask_rep]
print("步级训练样本:", Xs_tr.shape[0], "| 步级测试样本:", Xs_te.shape[0])

prm = make_mlp(7, hidden=64)
_ = train_mlp(prm, Xs_tr, ys_tr, iters=2500)
p_step = predict_prob(prm, Xs_te)
acc_step = ((p_step > 0.5) == ys_te).mean()
print("过程验证器步级准确率: {:.3f}".format(acc_step))
print("步级正例比例（基线）: {:.3f}".format(ys_te.mean()))
print("关键观察: 每步一个干净标签，过程验证器把每步的对错学了出来。")

In [ ]:
p_step_test = p_step.reshape(-1, 2)


def oracle_success(N):
    """完美过程验证器：用每步真值的乘积选解。"""
    ok = 0
    for pi in range(n_te):
        base = te_start + pi * cpp
        true_prod = [np.prod(pool[base + j]["step_ok"]) for j in range(N)]
        idx = base + int(np.argmax(true_prod))
        ok += int(te_answers[idx] == te_golden[idx])
    return ok / n_te


Ns = [1, 3, 5, 10, 15, 25]
print("N    多数    ORM(结果)  PRM(过程)  完美PRM")
for N in Ns:
    mj = majority_success(N)
    orm_ok = best_of_n_success(N, lambda b, N: p_raw[b:b + N])
    prm_ok = best_of_n_success(N, lambda b, N: p_step_test[b:b + N].prod(1))
    orc = oracle_success(N)
    print("{:<5} {:<6.2f} {:<9.2f} {:<9.2f} {:.2f}".format(
        N, mj, orm_ok, prm_ok, orc))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(Ns, [majority_success(N) for N in Ns], "o-", label="Majority vote")
plt.plot(Ns, [best_of_n_success(N, lambda b, N: p_raw[b:b + N]) for N in Ns],
         "s-", label="ORM (outcome)")
plt.plot(Ns, [best_of_n_success(N, lambda b, N: p_step_test[b:b + N].prod(1))
              for N in Ns], "^-", label="PRM (process)")
plt.plot(Ns, [oracle_success(N) for N in Ns], "d--", label="Perfect verifier")
plt.xlabel("Number of candidates N")
plt.ylabel("Success rate")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 挑一条"中间步错、最终答案对"的测试解，看它的过程分数
te_lucky = [d for d in pool[te_start:] if not all(d["step_ok"])
            and d["final_ok"]]
example = te_lucky[0]
k_ex = pool.index(example) - te_start
probs_ex = p_step_test[k_ex]
print("解文本:\n" + example["text"])
print("每步 PRM 分数:", np.round(probs_ex, 3))
print("乘积归约:", round(score_product(probs_ex), 3))
print("最小值归约:", round(score_min(probs_ex), 3))


rows = p_step_test[:12]
plt.figure(figsize=(6, 3.5))
plt.imshow(rows, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
plt.colorbar()
plt.xlabel("Step")
plt.ylabel("Solution index")
plt.title("Per-step correctness probability")
plt.tight_layout()
plt.show()

Math-Shepherd 要解决的是步级标签的获取成本：步级标签好用，但人工标每一步太贵，于是把标注流程自动化。
标注的角度也换了：不问这一步本身算得对不对，而问从这一步继续还能不能推出正确答案。这个角度来自 MCTS——用一个补全器（completer）从当前中间步续写 N 条后续路径，看多少条最终到达黄金答案。
用具体数字演示一遍。假设某一步之后，补全器续写了 N = 8 条完整解，其中 3 条最终答案等于黄金答案、5 条不等于。硬估计（HE）只看存在性：有一条续写到达正确答案就标 1，所以这一步标 1。软估计（SE）看频率：3 / 8 = 0.375，这一步标 0.375。
两种估计用同一个式子表达。记第 j 条续写的最终答案为 $a_j$，黄金答案为 $a^*$：

```text
HE: y_i = 1[存在 j 使 a_j = a*]
SE: y_i = (1/N) * Σ_j 1[a_j = a*]
```

N 越大，SE 越接近这一步真实的正确率，HE 却更容易出假阳性——只要续写里有一条蒙对，这一步就被标成 1，即使它本身是错的。作者发现 SE 的分布更接近人工标注，HE 在 N 增大时因假阳性而退化。下面用 mock 补全器复现这个规律。


In [ ]:
def roll_out(prefix_ok, comp_acc, rng):
    """从当前状态续写一条完整路径，返回最终答案是否等于黄金答案。"""
    p = comp_acc if prefix_ok else 0.05
    return bool(rng.random() < p)


def auto_label(prefix_ok, N, comp_acc, rng):
    """对某个中间步打标：续写 N 条路径，返回 (HE, SE)。"""
    hits = np.array([roll_out(prefix_ok, comp_acc, rng) for _ in range(N)])
    return int(hits.any()), float(hits.mean())


rng = np.random.default_rng(7)
n_steps = 2000
true_ok = np.array([bool(rng.random() < 0.5) for _ in range(n_steps)])
comp_acc = 0.8

agree_he_list, agree_se_list = [], []
Ns_roll = [1, 4, 8, 16]
for N in Ns_roll:
    he_arr, se_arr = [], []
    for ok in true_ok:
        he, se = auto_label(ok, N, comp_acc, rng)
        he_arr.append(he)
        se_arr.append(se)
    he_arr = np.array(he_arr)
    se_arr = np.array(se_arr)
    a_he = (he_arr == true_ok).mean()
    a_se = ((se_arr > 0.5) == true_ok).mean()
    agree_he_list.append(a_he)
    agree_se_list.append(a_se)
    print("N={}: HE 一致率 {:.3f} | SE(阈0.5) 一致率 {:.3f} | HE 正例率 {:.3f}"
          .format(N, a_he, a_se, he_arr.mean()))

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
plt.plot(Ns_roll, agree_he_list, "o-", label="HE agreement")
plt.plot(Ns_roll, agree_se_list, "s-", label="SE agreement")
plt.xlabel("Number of rollouts N")
plt.ylabel("Agreement with true labels")
plt.legend()
plt.tight_layout()
plt.show()

步级标签的价值不只在测试时选解，还能当作强化学习的奖励。
传统结果监督在整条解结束时给一个 reward；步级标签允许在每一步结束时各给一个 reward——这一步算对给正分，算错给负分。强化学习每一步结束都收到信号，比只在终点收到一个信号更容易学。
Math-Shepherd 用自动标注的步级标签做了 step-by-step PPO，效果优于同规模的 ORM-PPO。第 6 讲的 GRPO 延续的正是这个思路：用步级奖励训练推理模型。
下面先用 SE 分数做 best-of-N 选解，再看一个步级 reward 的小例子：整条解的 reward 是各步 reward 之和。


In [ ]:
def se_score_solution(text, prob, N, comp_acc, rng):
    """用 SE 标签当步级分数：每步续写 N 条，返回 SE 的乘积。"""
    step_ok_true, _ = judge_solution(text, prob)
    scores = []
    for k in range(2):
        prefix_ok = all(step_ok_true[:k + 1])
        _, se = auto_label(prefix_ok, N, comp_acc, rng)
        scores.append(se)
    return float(np.prod(scores))


def best_of_n_se(N_sel, N_roll):
    """用 SE 乘积分数做 best-of-N 选解。"""
    rng2 = np.random.default_rng(3)
    ok = 0
    for pi in range(n_te):
        base = te_start + pi * cpp
        scores = []
        for j in range(N_sel):
            d = pool[base + j]
            scores.append(se_score_solution(d["text"], d["prob"],
                                            N_roll, comp_acc, rng2))
        idx = base + int(np.argmax(scores))
        ok += int(te_answers[idx] == te_golden[idx])
    return ok / n_te


for N in [10, 25]:
    print("N={}: 多数投票 {:.2f} | SE 自动标注 best-of-N {:.2f}".format(
        N, majority_success(N), best_of_n_se(N, 8)))

# step-by-step 强化学习的想法：每个步结束给一个 reward，而不是只在末尾给
step_rewards = np.array([0.9, 0.4, 0.85])
print("\n每步 reward:", step_rewards)
print("整条解 reward 之和:", round(step_rewards.sum(), 2))

In [ ]:
def pass_at_k_curve(K):
    """每个测试题前 K 条候选里是否至少一条最终正确。"""
    ok = 0
    for pi in range(n_te):
        base = te_start + pi * cpp
        ok += int(te_final_ok[base:base + K].any())
    return ok / n_te


K_vals = [2, 4, 8, 16, 25, 40]
print("K     Pass@K   多数    PRM   gap(多数) gap(PRM)")
for K in K_vals:
    pk = pass_at_k_curve(K)
    mj = majority_success(K)
    prm_ok = best_of_n_success(K, lambda b, N: p_step_test[b:b + N].prod(1))
    print("{:<5} {:<8.2f} {:<6.2f} {:<6.2f} {:<9.2f} {:.2f}".format(
        K, pk, mj, prm_ok, pk - mj, pk - prm_ok))

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(K_vals, [pass_at_k_curve(K) for K in K_vals], "o-",
         label="Pass@K (oracle in pool)")
plt.plot(K_vals, [majority_success(K) for K in K_vals], "s-",
         label="Majority vote")
plt.plot(K_vals, [best_of_n_success(K, lambda b, N: p_step_test[b:b + N].prod(1))
                  for K in K_vals], "^-", label="PRM best-of-N")
plt.xlabel("K (candidates)")
plt.ylabel("Success rate")
plt.legend()
plt.tight_layout()
plt.show()

单个验证器不可靠时，把多个弱验证器集成起来。问题是没有标签，于是需要从验证器自身的行为里估计精度。Weaver 的思路是：验证器之间的同意程度里藏着各自的精度。
先从两个验证器建立直觉。设一条解的正确与否为 Y（1 对 0 错），验证器各自独立地对每条解打 1/0（判定对/错）。在 Y=1 的解上，验证器 k 打 1 的概率叫真阳性率 TPR_k；在 Y=0 的解上，它打 0 的概率叫真阴性率 TNR_k。这两个量就是验证器的精度。
精度不能直接观测，但可以统计一些量：验证器 k 打 1 的比例，以及验证器 i、j 同时打 1 的比例。记 $p = P(Y=1)$ 为解的正确率，由条件独立：

```text
P(S_k = 1)          = p·TPR_k + (1-p)·(1-TNR_k)
P(S_i = 1, S_j = 1) = p·TPR_i·TPR_j + (1-p)·(1-TNR_i)·(1-TNR_j)
```

左边是能直接统计的观测值，右边是含未知数的模型。未知数包括共享的 p 和每个验证器的 TPR_k、TNR_k：两个验证器是 5 个未知数、只有 3 个方程，还解不出；验证器增加到 4 个以上，方程数（K 个单变量矩加 K(K-1)/2 个成对矩）就超过未知数（1 + 2K），用最小二乘可以把每个 TPR、TNR 都解出来。这就是矩估计——用可观测的矩反推不可观测的精度参数，全程不需要人工标签。
有了精度，选解变成概率推断。给定一条解，各验证器打了向量 s，朴素贝叶斯假设各验证器在正确性条件下独立，算出后验：

```text
P(Y=1 | s) ∝ P(Y=1) · Π_k P(S_k = s_k | Y=1)
```

其中 $P(S_k=1|Y=1) = TPR_k$、$P(S_k=0|Y=1) = 1 - TPR_k$。分数最高的解，就是被一堆可靠验证器共同看好的解。下面代码模拟 5 个不同精度的弱验证器，只用无标签的同意率估计 TPR/TNR，再按后验选解。


In [ ]:
from scipy.optimize import least_squares

K_v = 5
true_tpr = np.array([0.62, 0.71, 0.80, 0.55, 0.88])
true_tnr = np.array([0.58, 0.66, 0.75, 0.52, 0.84])

# 在测试解上模拟 5 个弱验证器的投票
rng = np.random.default_rng(11)
Y_te = te_final_ok[te_start:]
n_te_sol = len(Y_te)
S = np.zeros((n_te_sol, K_v))
for k in range(K_v):
    p = np.where(Y_te == 1, true_tpr[k], 1 - true_tnr[k])
    S[:, k] = (rng.random(n_te_sol) < p).astype(float)

obs1 = S.mean(0)
obs2 = (S[:, :, None] * S[:, None, :]).mean(0)
pairs = [(i, j) for i in range(K_v) for j in range(i + 1, K_v)]


def residuals(theta):
    """矩方程残差：单变量矩 + 成对矩。"""
    pY, tpr, tnr = theta[0], theta[1:1 + K_v], theta[1 + K_v:]
    r = [pY * tpr[k] + (1 - pY) * (1 - tnr[k]) - obs1[k]
         for k in range(K_v)]
    for i, j in pairs:
        r.append(pY * tpr[i] * tpr[j] +
                 (1 - pY) * (1 - tnr[i]) * (1 - tnr[j]) - obs2[i, j])
    return np.array(r)


x0 = np.concatenate([[0.3], np.full(2 * K_v, 0.7)])
res = least_squares(residuals, x0, bounds=(0.001, 0.999), max_nfev=20000)
pY_hat, tpr_hat, tnr_hat = res.x[0], res.x[1:1 + K_v], res.x[1 + K_v:]
print("P(Y) 估计: {:.3f}（真实 {:.3f}）".format(pY_hat, Y_te.mean()))
print("TPR 估计:", np.round(tpr_hat, 2), "（真实", true_tpr, "）")
print("TNR 估计:", np.round(tnr_hat, 2), "（真实", true_tnr, "）")


def posterior_score(s, pY, tpr, tnr):
    """朴素贝叶斯后验 P(Y=1|s)。"""
    p1 = pY * np.prod(np.where(s == 1, tpr, 1 - tpr))
    p0 = (1 - pY) * np.prod(np.where(s == 1, 1 - tnr, tnr))
    return p1 / (p1 + p0)


def weaver_success(N, use):
    """用不同聚合方式选解，比较命中率。"""
    ok = 0
    for pi in range(n_te):
        base = pi * cpp
        votes = S[base:base + N]
        if use == "majority":
            vals = te_answers[te_start + base:te_start + base + N]
            counts = {}
            for v in vals:
                counts[v] = counts.get(v, 0) + 1
            idx = int(np.argmax([counts.get(v, 0) for v in vals]))
        elif use == "equal":
            idx = int(np.argmax(votes.mean(1)))
        elif use == "mom":
            sc = np.array([posterior_score(votes[j], pY_hat, tpr_hat, tnr_hat)
                           for j in range(N)])
            idx = int(np.argmax(sc))
        else:
            sc = np.array([posterior_score(votes[j], Y_te.mean(),
                                           true_tpr, true_tnr)
                           for j in range(N)])
            idx = int(np.argmax(sc))
        ok += int(te_answers[te_start + base + idx]
                  == te_golden[te_start + base + idx])
    return ok / n_te


for N in [10, 25]:
    row = " | ".join(
        "{} {:.2f}".format(s, weaver_success(N, s))
        for s in ["majority", "equal", "mom", "oracle-w"])
    print("N={}: {}".format(N, row))

In [ ]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()


CN_OP = {"+": "加", "-": "减", "*": "乘以"}


def llm_sub_answer(client, a, op, b):
    """让 LLM 算一个子表达式，返回整数；解析不到返回 None。"""
    prompt = "请计算 {} {} {} 等于多少，只回答数字。".format(a, CN_OP[op], b)
    try:
        reply = client.chat([{"role": "user", "content": prompt}])
    except Exception:
        reply = "模拟输出：无法连接 API。"
    m = re.search(r"-?\d+", reply)
    return int(m.group(0)) if m else None


def llm_propose_full(client, problem):
    """让 LLM 分两步算两个子式，组合出候选最终答案。"""
    (a1, op1, b1), (a2, op2, b2) = problem["parts"]
    l = llm_sub_answer(client, a1, op1, b1)
    r = llm_sub_answer(client, a2, op2, b2)
    if l is None or r is None:
        return None, None
    ans = apply(problem["combine"], l, r)
    text = "\n".join([
        f"{a1} {op1} {b1} = {l}",
        f"{a2} {op2} {b2} = {r}",
        f"答案 = {ans}",
    ])
    return text, ans


# 挑三道只含加减法的题，避免 mock 无法处理乘法
simple_probs = []
seen = set()
for d in pool:
    parts = tuple(sorted(d["prob"]["parts"]))
    if parts in seen:
        continue
    if all(op in "+-" for (_, op, _) in d["prob"]["parts"]):
        seen.add(parts)
        simple_probs.append(d["prob"])
    if len(simple_probs) == 3:
        break

print("当前为 mock 模式：LLM 对加减法给出确定性结果；真实 API 下模型可能算错，由验证器拦截。")
for prob in simple_probs:
    text, ans = llm_propose_full(client, prob)
    if text is None:
        print("无法解析 LLM 输出，跳过。")
        continue
    _, final_ok = judge_solution(text, prob)
    score = predict_prob(orm, orm_features(text).reshape(1, -1))[0]
    print("题面:", prob["parts"], "| 黄金:", prob["golden"])
    print("LLM 提议答案:", ans, "| 判定:", "对" if final_ok else "错",
          "| 结果验证器分数: {:.3f}".format(score))

In [ ]:
# 作业 1：补全两种归约
step_probs = np.array([0.9, 0.8, 0.2, 0.9])


def score_product(probs):
    """乘积归约：所有步概率相乘。"""
    return float(np.prod(probs))  # 填空处：np.prod


def score_min(probs):
    """最小值归约：取概率最小值。"""
    return float(np.min(probs))   # 填空处：np.min


assert abs(score_min(step_probs) - 0.2) < 1e-9
assert abs(score_product(step_probs) - 0.9 * 0.8 * 0.2 * 0.9) < 1e-9
print("两种归约都正确：乘积归约惩罚步数多的解，最小值归约只盯最差的一步。")

In [ ]:
# 作业 2：补全 HE 与 SE
answers = np.array([True, True, False, False, True, False, False, False])


def hard_estimation(flags):
    """硬估计：只要有一条续写到达正确答案就返回 1。"""
    return int(flags.any())         # 填空处：any


def soft_estimation(flags):
    """软估计：返回到达正确答案的续写比例。"""
    return float(flags.mean())      # 填空处：mean


assert hard_estimation(answers) == 1
assert abs(soft_estimation(answers) - 3 / 8) < 1e-9
print("HE 与 SE 都正确：SE 是频率、HE 是存在性，N 越小 SE 噪声越大。")

In [ ]:
# 作业 3：补全两种选择器
candidates = [
    (0.9, "A", True), (0.85, "A", True),
    (0.1, "B", False), (0.15, "B", False),
    (0.2, "B", False), (0.05, "B", False),
]


def choose_by_verifier(cands):
    """验证器选最高分那条的最终答案。"""
    best = max(cands, key=lambda c: c[0])  # 填空处：c[0] 取分数
    return best[1]


def choose_by_majority(cands):
    """多数投票选出现最多的最终答案。"""
    from collections import Counter
    counts = Counter(c[1] for c in cands)  # 填空处：c[1] 取答案
    return max(counts, key=counts.get)


assert choose_by_verifier(candidates) == "A"
assert choose_by_majority(candidates) == "B"
print("验证器选 A（分数高），多数投票选 B（出现多）：错误答案扎堆但分数低时，验证器胜出。")